# 04 - Preprocesamiento

Preparamos los datos para el modelo resolviendo tres problemas
detectados en el EDA: multicolinealidad, distribuciones asimétricas
y diferencias de escala entre features.

## 1. Carga de datos desde Supabase

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from supabase import create_client
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os

load_dotenv(dotenv_path='../.env')

supabase = create_client(
    os.getenv("SUPABASE_URL"),
    os.getenv("SUPABASE_KEY")
)

todos = []
page = 0
page_size = 1000

while True:
    response = supabase.table('asteroids')\
        .select('*')\
        .range(page * page_size, (page + 1) * page_size - 1)\
        .execute()
    if not response.data:
        break
    todos.extend(response.data)
    page += 1

df = pd.DataFrame(todos)
print(f"Datos cargados: {df.shape[0]} filas, {df.shape[1]} columnas")

Datos cargados: 11775 filas, 9 columnas


## 2. Ingeniería de features

Combinamos diameter_min_km y diameter_max_km en una sola variable.
El EDA nos mostró una correlación de 1.00, tener ambas es redundante.

In [2]:
# Creamos el promedio entre min y max
df['diameter_mean_km'] = (df['diameter_min_km'] + df['diameter_max_km']) / 2

# Eliminamos las dos originales
df = df.drop(columns=['diameter_min_km', 'diameter_max_km'])

print("Columnas actuales:")
print(df.columns.tolist())

Columnas actuales:
['id', 'name', 'absolute_magnitude_h', 'velocity_km_per_hour', 'miss_distance_km', 'close_approach_date', 'is_potentially_hazardous', 'diameter_mean_km']


## 3. Transformación logarítmica

Las features con distribución asimétrica (cola larga hacia la derecha)
pueden distorsionar el aprendizaje del modelo. Aplicamos log para 
"aplanar" esa asimetría.

np.log1p() aplica log(x + 1) en lugar de log(x) para evitar 
problemas con valores iguales a cero.

In [3]:
features_a_transformar = ['diameter_mean_km', 'velocity_km_per_hour', 'miss_distance_km']

for feature in features_a_transformar:
    df[f'{feature}_log'] = np.log1p(df[feature])
    df = df.drop(columns=[feature])

print("Columnas después de transformación logarítmica:")
print(df.columns.tolist())

Columnas después de transformación logarítmica:
['id', 'name', 'absolute_magnitude_h', 'close_approach_date', 'is_potentially_hazardous', 'diameter_mean_km_log', 'velocity_km_per_hour_log', 'miss_distance_km_log']


## 4. Separación de features y target

"X" contiene las variables que el modelo va a usar para aprender.
"y" contiene lo que el modelo tiene que predecir.

Eliminamos id, name y close_approach_date porque no son features 
útiles para el modelo, son identificadores o metadata.

In [4]:
X = df.drop(columns=['id', 'name', 'close_approach_date', 'is_potentially_hazardous'])
y = df['is_potentially_hazardous']

print("Features (X):")
print(X.columns.tolist())
print(f"\nDimensiones de X: {X.shape}")
print(f"Dimensiones de y: {y.shape}")
print(f"\nDistribución del target:")
print(y.value_counts())

Features (X):
['absolute_magnitude_h', 'diameter_mean_km_log', 'velocity_km_per_hour_log', 'miss_distance_km_log']

Dimensiones de X: (11775, 4)
Dimensiones de y: (11775,)

Distribución del target:
is_potentially_hazardous
False    11215
True       560
Name: count, dtype: int64


## 5. Split train/test

Dividimos el dataset en dos partes:
- **Train (80%):** los datos con los que el modelo aprende
- **Test (20%):** los datos que el modelo nunca vio, usados solo para evaluar

Esto es fundamental — si evaluamos el modelo con los mismos datos 
con los que aprendió, no sabemos si realmente aprendió o si solo 
memorizó. Separar test simula cómo va a funcionar el modelo en el 
mundo real.

stratify=y garantiza que la proporción de peligrosos/no peligrosos 
sea la misma en train y test. Sin est

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y # ayuda a que quede la misma proporción en ambos sets
)

print(f"Train: {X_train.shape[0]} filas")
print(f"Test:  {X_test.shape[0]} filas")
print(f"\nDistribución en train:")
print(y_train.value_counts())
print(f"\nDistribución en test:")
print(y_test.value_counts())

Train: 9420 filas
Test:  2355 filas

Distribución en train:
is_potentially_hazardous
False    8972
True      448
Name: count, dtype: int64

Distribución en test:
is_potentially_hazardous
False    2243
True      112
Name: count, dtype: int64


In [6]:
print("Proporción en train:")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nProporción en test:")
print((y_test.value_counts(normalize=True) * 100).round(2))

Proporción en train:
is_potentially_hazardous
False    95.24
True      4.76
Name: proportion, dtype: float64

Proporción en test:
is_potentially_hazardous
False    95.24
True      4.76
Name: proportion, dtype: float64


## 6. Escalado de features

Escalamos para que todas las features estén en el mismo rango.
Sin esto, variables con números grandes como miss_distance_km_log 
dominarían el aprendizaje sobre variables con números chicos como 
absolute_magnitude_h.

StandardScaler transforma cada feature para que tenga media 0 
y desvío estándar 1.

IMPORTANTE: el scaler se entrena SOLO con X_train y se aplica 
a ambos. Si lo entrenáramos con todo el dataset, el modelo 
"vería" información del test antes de tiempo (data leakage).

In [7]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print("Media de cada feature en train (debe ser ~0):")
print(X_train_scaled.mean().round(4))
print("\nDesvío estándar de cada feature en train (debe ser ~1):")
print(X_train_scaled.std().round(4))

Media de cada feature en train (debe ser ~0):
absolute_magnitude_h       -0.0
diameter_mean_km_log       -0.0
velocity_km_per_hour_log    0.0
miss_distance_km_log       -0.0
dtype: float64

Desvío estándar de cada feature en train (debe ser ~1):
absolute_magnitude_h        1.0001
diameter_mean_km_log        1.0001
velocity_km_per_hour_log    1.0001
miss_distance_km_log        1.0001
dtype: float64


## Resumen

- Combinamos diameter_min_km y diameter_max_km en diameter_mean_km
- Aplicamos log1p a diameter, velocity y miss_distance
- Split 80/20 estratificado por clase
- StandardScaler entrenado solo con X_train

Datos listos para modelar en 05_models.ipynb.